In [1]:
import os

### Write ```Dockerfile```

In [2]:
%%writefile Dockerfile

FROM python:3.9
    
# update package manager
RUN apt-get update

# update pip
RUN pip install --upgrade pip

# copy requirements
COPY requirements.txt .

# install dependencies
RUN pip install -r requirements.txt

# copy script into container
COPY script.py .

# run script when image is run
CMD ["python3", "script.py"]

Writing Dockerfile


### Write ```requirements.txt``` to local drive

In [3]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

pandas==1.2.4
catboost==1.0.4
scikit_learn==0.24.1

Writing requirements.txt


### Write ```script.py``` to local drive

In [4]:
%%writefile script.py

import os
import pandas as pd
import numpy as np
import catboost as cb
import sklearn.metrics as skm

# get metric
def get_metric_by_year_month(target, y_hat, str_eval_metric):
    if str_eval_metric == 'AUC':
        return skm.roc_auc_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'PRAUC':
        return skm.average_precision_score(y_true=target, y_score=y_hat)
    elif str_eval_metric == 'Logloss':
        return skm.log_loss(y_true=target, y_pred=y_hat)
    elif str_eval_metric == 'F1':
        return skm.f1_score(y_true=target, y_pred=y_hat)
    else:
        pass

# get index of array
int_idx_array = int(os.environ['AWS_BATCH_JOB_ARRAY_INDEX'])
print(f'Array index: {int_idx_array}')

# constants
str_project = '20231010-gen-xii'
str_target = 'target'
str_model = '02_pricing_pd'

###############################################################################
# HYPERPARAMETERS
###############################################################################
# get df_hyperparameters
str_filename = 'df_hyperparameters.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/05_step_function/{str_filename}'
df = pd.read_csv(str_uri)
# convert to dict
dict_hyperparameters = dict(zip(df['keys'], df['values']))

# get number of iterations
int_n_iterations = int(dict_hyperparameters['INT_N_ITERATIONS'])
print(f'Iterations: {int_n_iterations}')

# get filename for training
str_filename_train = dict_hyperparameters['STR_FILENAME_TRAIN']
print(f'Training filename: {str_filename_train}')

# get filename for valid
str_filename_valid = dict_hyperparameters['STR_FILENAME_VALID']
print(f'Valid filename: {str_filename_valid}')

# number of tuning jobs
int_n_tuning_jobs = int(dict_hyperparameters['INT_N_TUNING_JOBS_1'])
print(f'Total tuning jobs: {int_n_tuning_jobs}')

# get proportion of iterations to use as early stopping rounds
flt_prop_early_stopping = float(dict_hyperparameters['PROP_EARLY_STOPPING'])
print(f'Proportion early stopping: {flt_prop_early_stopping}')

# get eval metric
str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
print(f'Eval metric: {str_eval_metric}')

##################################################################################

# get list of features
print('Importing df_list_features.csv...')
str_filename = 'df_list_features.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/01_lambda_get_starting_feats/{str_filename}'
list_cols_model = list(pd.read_csv(str_uri)['feature'])
list_cols_model = [col for col in list_cols_model if col != str_target]

# get lr
print('Getting learning rate...')
flt_learning_rate = list(np.around(np.linspace(0.001, 0.999, int_n_tuning_jobs), 4))[int_idx_array] # round each value to 4 decimal places

# read training data
print('Reading training data...')
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri)

# make sure all cols in list_cols_model are in df
list_cols_model = [col for col in list_cols_model if col in list(df.columns)]

# class weights
print('Getting class weights...')
# these are coming from the test data set in gen 12 v2
flt_desired_0_prop = 0.6974
flt_desired_1_prop = 0.3026
# get the current proportions
ser_prop = df[str_target].value_counts(normalize=True)
# get currnt percentages
flt_current_0_prop = ser_prop[0]
flt_current_1_prop = ser_prop[1]
# get the weights
flt_0_weight = flt_desired_0_prop / flt_current_0_prop
flt_1_weight = flt_desired_1_prop / flt_current_1_prop
list_class_weights = [flt_0_weight, flt_1_weight]

# get the non numeric feats
print('Getting list of non-numeric columns...')
list_cols_non_numeric = []
for col in list_cols_model:
    if df[col].dtype not in ['float64','int64']:
        list_cols_non_numeric.append(col)

# build model
print('Building model...')
# pool data
pool_train = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# read validation data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri)

# pool data
pool_valid = cb.Pool(
    df[list_cols_model], 
    df[str_target], 
    cat_features=list_cols_non_numeric,
)
del df

# constraints
dict_monotone_constraints = {
    # better
    'fltgrossmonthly__income_sum': -1, # as income increases, prediction gets better
    'fltapproveddowntotal__app': -1,
    'fltdowncash__app': -1,
    'bookvalue__app': -1,
    'ENG-dealership_age': -1,
    'bookvalue__app': -1,
    'fltdowncash__app': -1,
    'fltapproveddowntotal__app': -1,
    # worse
    'fltgrossmonthly__income_count': 1, # as count of income increases, prediction gets worse
    'ENG-loan_to_value': 1,
    'ENG-payment_to_income': 1,
    'ENG-vehicle_age': 1,
    'fltadvance__app': 1,
    'bigmileage_odometer__app': 1,
    'amtfinanced__app': 1,
    'miles_odometer__app': 1,
    'pti__app': 1,
    'fltadvance__app': 1,
    'ENG-loan_to_value': 1,
    'amtfinanced__app': 1,
}
# ensure features are in list_cols_model
dict_monotone_constraints = {key: val for key, val in dict_monotone_constraints.items() if key in list_cols_model}

# init class
cls_model_inference = cb.CatBoostClassifier(
    task_type='CPU',
    nan_mode='Min',
    random_state=42,
    eval_metric=str_eval_metric,
    iterations=int_n_iterations,
    learning_rate=flt_learning_rate,
    class_weights=list_class_weights,
    monotone_constraints=dict_monotone_constraints,
)

# fit
cls_model_inference.fit(
    pool_train,
    eval_set=[pool_valid],
    verbose=100,
    use_best_model=True,
    early_stopping_rounds=int(round(int_n_iterations*flt_prop_early_stopping)), 
)
del pool_train
del pool_valid

################################################################################################
# GET TRAINING EVAL METRIC
################################################################################################
print('Getting training eval metric...')

# import data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_train}'
df = pd.read_parquet(str_uri)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - train
if str_eval_metric == 'AUC':
    flt_eval_metric_train = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_train = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_train = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_train = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# GET VALIDATION EVAL METRIC (WEIGHTED)
################################################################################################
print('Getting validation eval metric...')

# import data
str_uri = f's3://{str_project}/{str_model}/02_model/00_preprocessing/02_make_dfs/{str_filename_valid}'
df = pd.read_parquet(str_uri)

# get predictions
if str_eval_metric in ['AUC','PRAUC','Logloss']:
    # probabilities
    df['y_hat'] = cls_model_inference.predict_proba(df[cls_model_inference.feature_names_])[:,1]
elif str_eval_metric in ['F1']:
    # class
    df['y_hat'] = cls_model_inference.predict(df[cls_model_inference.feature_names_])
else:
    pass

# get eval metric - valid
if str_eval_metric == 'AUC':
    flt_eval_metric_valid = skm.roc_auc_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'PRAUC':
    flt_eval_metric_valid = skm.average_precision_score(y_true=df[str_target], y_score=df['y_hat'])
elif str_eval_metric == 'Logloss':
    flt_eval_metric_valid = skm.log_loss(y_true=df[str_target], y_pred=df['y_hat'])
elif str_eval_metric == 'F1':
    flt_eval_metric_valid = skm.f1_score(y_true=df[str_target], y_pred=df['y_hat'])

# save memory
del df

################################################################################################
# CREATE OUTPUT DATA FRAME
################################################################################################
print('Creating output data frame...')

flt_diff = abs(flt_eval_metric_train - flt_eval_metric_valid)
dict_row = {
    'iteration': int_idx_array,
    'learning_rate': flt_learning_rate,
    'flt_eval_metric_train': flt_eval_metric_train,
    'flt_eval_metric_valid': flt_eval_metric_valid,
    'diff': flt_diff,
    'best_iteration': cls_model_inference.get_best_iteration(),
}
df = pd.DataFrame(dict_row, index=[0])

# save
str_filename = f'df_output_{int_idx_array}.csv'
str_uri = f's3://{str_project}/{str_model}/02_model/01_feat_select/02_batch_tuning/models/{str_filename}'
df.to_csv(str_uri, index=False)

Overwriting script.py


### Build and push to ECR

In [5]:
%%sh

# name the image
image=genxii-pd-tuning-1

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  96.26kB
Step 1/7 : FROM python:3.9
 ---> ab7eeae5d25f
Step 2/7 : RUN apt-get update
 ---> Using cache
 ---> 4ff3e2930093
Step 3/7 : RUN pip install --upgrade pip
 ---> Using cache
 ---> d4e484328ae0
Step 4/7 : COPY requirements.txt .
 ---> Using cache
 ---> 2b6a8272998d
Step 5/7 : RUN pip install -r requirements.txt
 ---> Using cache
 ---> e2c5362840d3
Step 6/7 : COPY script.py .
 ---> abc483f81015
Step 7/7 : CMD ["python3", "script.py"]
 ---> Running in 157ca31f8cb5
Removing intermediate container 157ca31f8cb5
 ---> 5aadf693609b
Successfully built 5aadf693609b
Successfully tagged genxii-pd-tuning-1:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-pd-tuning-1' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-pd-tuning-1]
7117f7e2eaed: Preparing
92ce049e7833: Preparing
74c21452c9ee: Preparing
ebd2f2f1c999: Preparing
d2329cd063ca: Preparing
47e31a4d606a: Preparing
bf4966b4b813: Preparing
da15a2a37253: Preparing
89ca33c95b2e: Preparing
83db175c22e2: Preparing
c5d13b2949a2: Preparing
7e43f593c900: Preparing
072686bcd3db: Preparing
da15a2a37253: Waiting
89ca33c95b2e: Waiting
bf4966b4b813: Waiting
c5d13b2949a2: Waiting
47e31a4d606a: Waiting
83db175c22e2: Waiting
7e43f593c900: Waiting
072686bcd3db: Waiting
74c21452c9ee: Layer already exists
92ce049e7833: Layer already exists
ebd2f2f1c999: Layer already exists
d2329cd063ca: Layer already exists
47e31a4d606a: Layer already exists
bf4966b4b813: Layer already exists
89ca33c95b2e: Layer already exists
da15a2a37253: Layer already exists
83db175c22e2: Layer already exists
c5d13b2949a2: Layer already exists
7e43f593c900: Layer already exists
072686bcd3db: Layer already exi

### Clean-up

In [6]:
# rm files
for str_file in ['Dockerfile','requirements.txt','script.py']:
    try:
        os.remove(f'./{str_file}')
    except:
        pass